# 02 · Silver — Clean, typed & deduplicated trips

Consumes bronze **Change Data Feed** and upserts cleaned, conformed trips into the
silver Delta table.

- Parses the multiple historical timestamp formats and casts lat/lng to `double`.
- Derives `trip_duration_mins` (nulls out impossible > 24 h trips) and normalises
  `member_casual` (`Subscriber`/`Customer` → `member`/`casual`).
- Filters invalid rows, then **`MERGE`**s the latest change per `ride_id`
  (`foreachBatch`), making the load idempotent and delete-aware.

**Upstream:** `01_bronze` · **Downstream:** `03_gold`

In [ ]:
import sys
from pathlib import Path

from pyspark.sql.functions import (
    coalesce,
    col,
    create_map,
    current_timestamp,
    date_format,
    expr,
    lit,
    row_number,
    to_date,
    when,
)
from pyspark.sql.window import Window

sys.path.append(str(Path.cwd().parent.parent))

env_name = dbutils.widgets.get("env") if dbutils.widgets.get("env") else "dev"
pipeline_id = dbutils.widgets.get("pipeline_id") if dbutils.widgets.get("pipeline_id") else "placeholder"
job_run_id = dbutils.widgets.get("job_run_id") if dbutils.widgets.get("job_run_id") else "placeholder"
task_id = dbutils.widgets.get("task_run_id") if dbutils.widgets.get("task_run_id") else "placeholder"

catalog_name = f"citibike_ext_{env_name}" if env_name == "dev" else f"citibike_{env_name}"
checkpoint_path = f"/Volumes/{catalog_name}/02_silver/_checkpoint/citibike_trips/"
bronze_table = f"{catalog_name}.01_bronze.citibike_trips"
silver_table = f"{catalog_name}.02_silver.citibike_trips"

# Read row-level changes from bronze via Change Data Feed.
df = spark.readStream.format("delta").option("readChangeFeed", "true").table(bronze_table)

create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {silver_table} (
    ride_id            STRING,
    trip_start_date    DATE,
    started_at         TIMESTAMP,
    ended_at           TIMESTAMP,
    rideable_type      STRING,
    start_station_id   STRING,
    start_station_name STRING,
    end_station_id     STRING,
    end_station_name   STRING,
    trip_duration_mins DOUBLE,
    start_lat          DOUBLE,
    start_lng          DOUBLE,
    end_lat            DOUBLE,
    end_lng            DOUBLE,
    member_casual      STRING,
    metadata           MAP<STRING, STRING>
) TBLPROPERTIES (delta.enableChangeDataFeed = true);
"""
spark.sql(create_table_sql)

df = df.withColumns(
    {
        "started_at": coalesce(
            expr("try_to_timestamp(started_at, 'yyyy-MM-dd HH:mm:ss.SSS')"),
            expr("try_to_timestamp(started_at, 'yyyy-MM-dd HH:mm:ss')"),
            expr("try_to_timestamp(started_at, 'M/d/yyyy HH:mm:ss')"),
        ),
        "ended_at": coalesce(
            expr("try_to_timestamp(ended_at, 'yyyy-MM-dd HH:mm:ss.SSS')"),
            expr("try_to_timestamp(ended_at, 'yyyy-MM-dd HH:mm:ss')"),
            expr("try_to_timestamp(ended_at, 'M/d/yyyy HH:mm:ss')"),
        ),
        "start_lat": col("start_lat").cast("double"),
        "start_lng": col("start_lng").cast("double"),
        "end_lat": col("end_lat").cast("double"),
        "end_lng": col("end_lng").cast("double"),
    }
)

df = df.withColumns(
    {
        "trip_start_date": to_date("started_at"),
        "trip_duration_mins": when(
            (col("ended_at").cast("long") - col("started_at").cast("long")) / 60 > 1440,  # >24h
            None,
        ).otherwise((col("ended_at").cast("long") - col("started_at").cast("long")) / 60),
        "member_casual": when(col("member_casual").isin("Subscriber", "member"), "member")
        .when(col("member_casual").isin("Customer", "casual"), "casual")
        .otherwise(col("member_casual")),
        "metadata": create_map(
            lit("pipeline_id"),
            lit(pipeline_id),
            lit("run_id"),
            lit(job_run_id),
            lit("task_id"),
            lit(task_id),
            lit("processed_timestamp"),
            date_format(current_timestamp(), "yyyy-MM-dd'T'HH:mm:ss"),
        ),
    }
)

df = df.filter(
    col("ride_id").isNotNull()
    & col("started_at").isNotNull()
    & col("ended_at").isNotNull()
    & (col("ended_at") >= col("started_at"))
)

df = df.select(
    "ride_id",
    "trip_start_date",
    "started_at",
    "ended_at",
    "rideable_type",
    "start_station_id",
    "start_station_name",
    "end_station_id",
    "end_station_name",
    "trip_duration_mins",
    "start_lat",
    "start_lng",
    "end_lat",
    "end_lng",
    "member_casual",
    "metadata",
    "_change_type",
    "_commit_version",
    "_commit_timestamp",  # keep these for the merge
)


def upsert_to_silver(batch_df, batch_id):
    # Keep only the latest change per ride_id, dropping pre-images and deletes,
    # then MERGE the resulting rows into the silver table.
    w = Window.partitionBy("ride_id").orderBy(col("_commit_version").desc())
    (
        batch_df.filter("_change_type IN ('insert', 'update_postimage')")
        .withColumn("_rn", row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn", "_change_type", "_commit_version", "_commit_timestamp")
        .createOrReplaceTempView("trip_changes")
    )

    batch_df.sparkSession.sql(f"""
        MERGE INTO {silver_table} AS t
        USING trip_changes AS s
        ON t.ride_id = s.ride_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)


(
    df.writeStream.option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .foreachBatch(upsert_to_silver)
    .start()
)